# Erdos-Straus Conjecture Solver

**Conjecture:** For every integer n >= 2, the fraction 4/n can be written as  
4/n = 1/x + 1/y + 1/z for positive integers x <= y <= z.

**Project:** [github.com/Commencethescourge/erdos-straus-solver](https://github.com/Commencethescourge/erdos-straus-solver)  
**Company:** Guinea Pig Trench LLC  

---

### Strategy

Only values of n where `n mod 24` is in `{1, 17}` require brute-force search ("hard residues").  
All other residues have known closed-form decompositions.  
We have already verified the conjecture up to **100,000,000**.  

This notebook continues the search from 100M onward, leveraging Kaggle's 4-core CPU,  
~29 GB RAM, and 12-hour background execution sessions.

### Usage

1. Edit the range in **Cell 4** to choose your search window.
2. Run all cells.
3. Close the tab if desired — Kaggle background execution keeps it running.
4. Results are saved to `/kaggle/working/` and persist as a dataset artifact.

In [ ]:
# Cell 2: Setup — detect cores, RAM, configure workers
import os
import math
import csv
import time
import psutil
from multiprocessing import Pool, cpu_count
from datetime import datetime, timedelta

NUM_CORES = cpu_count()
RAM_GB = psutil.virtual_memory().total / (1024 ** 3)
WORKERS = min(4, NUM_CORES)  # Kaggle gives 4 CPU cores

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"CPU cores detected: {NUM_CORES}")
print(f"RAM available:      {RAM_GB:.1f} GB")
print(f"Workers to use:     {WORKERS}")
print(f"Output directory:   {OUTPUT_DIR}")

In [ ]:
# Cell 3: Solver code (self-contained)

HARD_RESIDUES = {1, 17}


def solve_single(n, step_cap=20_000_000, y_cap_per_x=2_000_000):
    if n <= 1:
        return None
    steps = 0
    x_min = max(1, math.ceil(n / 4))
    x_max = n
    for x in range(x_min, x_max + 1):
        num_r = 4 * x - n
        if num_r <= 0:
            steps += 1
            if steps >= step_cap:
                return None
            continue
        den_r = n * x
        y_min = math.ceil(den_r / num_r)
        y_max = 2 * den_r // num_r
        y_steps = 0
        for y in range(max(x, y_min), y_max + 1):
            steps += 1
            y_steps += 1
            if steps >= step_cap:
                return None
            if y_steps >= y_cap_per_x:
                break
            denom_z = num_r * y - den_r
            if denom_z <= 0:
                continue
            num_z = den_r * y
            if num_z % denom_z == 0:
                z = num_z // denom_z
                if z >= y:
                    return {"n": n, "x": x, "y": y, "z": z, "steps": steps}
    return None


def generate_hard_residues(start, end):
    """Generate only n values where n mod 24 is in {1, 17}."""
    candidates = []
    for n in range(start, end + 1):
        if n % 24 in HARD_RESIDUES:
            candidates.append(n)
    return candidates


print(f"Solver loaded. Hard residues: {HARD_RESIDUES}")
print(f"Step cap per n: 20,000,000 | y cap per x: 2,000,000")

In [ ]:
# Cell 4: Range generator (USER-EDITABLE)
# ============================================================
# Edit these values to set your search range.
# We've verified up to 100M, so default starts at 100,000,001.
# ============================================================

RANGE_START = 100_000_001
RANGE_END   = 101_000_000

# ============================================================

candidates = generate_hard_residues(RANGE_START, RANGE_END)
print(f"Range: {RANGE_START:,} to {RANGE_END:,}")
print(f"Hard-residue candidates to solve: {len(candidates):,}")
print(f"(Skipped {RANGE_END - RANGE_START + 1 - len(candidates):,} values with known decompositions)")

In [ ]:
# Cell 5: Solver runner with progress output and CSV writing

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename = f"erdos_straus_{RANGE_START}_{RANGE_END}_{timestamp}.csv"
csv_path = os.path.join(OUTPUT_DIR, csv_filename)
unsolved_path = os.path.join(OUTPUT_DIR, f"unsolved_{RANGE_START}_{RANGE_END}_{timestamp}.txt")

print(f"Results CSV:  {csv_path}")
print(f"Unsolved log: {unsolved_path}")
print(f"Workers: {WORKERS}")
print("="*60)

BATCH_SIZE = 500
total = len(candidates)
solved_count = 0
unsolved_count = 0
total_steps = 0
unsolved_ns = []

t_start = time.time()

with open(csv_path, "w", newline="") as csvfile, \
     open(unsolved_path, "w") as unsolved_file:

    writer = csv.DictWriter(csvfile, fieldnames=["n", "x", "y", "z", "steps"])
    writer.writeheader()

    with Pool(processes=WORKERS) as pool:
        for batch_start in range(0, total, BATCH_SIZE):
            batch = candidates[batch_start : batch_start + BATCH_SIZE]
            results = pool.map(solve_single, batch)

            for result in results:
                if result is not None:
                    solved_count += 1
                    total_steps += result["steps"]
                    writer.writerow(result)
                else:
                    unsolved_count += 1

            # Find unsolved n values in this batch
            for n_val, result in zip(batch, results):
                if result is None:
                    unsolved_ns.append(n_val)
                    unsolved_file.write(f"{n_val}\n")

            csvfile.flush()
            unsolved_file.flush()

            done = batch_start + len(batch)
            elapsed = time.time() - t_start
            rate = done / elapsed if elapsed > 0 else 0
            eta = (total - done) / rate if rate > 0 else 0

            print(
                f"[{done:,}/{total:,}] "
                f"solved={solved_count:,} unsolved={unsolved_count:,} "
                f"rate={rate:.0f}/s "
                f"elapsed={timedelta(seconds=int(elapsed))} "
                f"ETA={timedelta(seconds=int(eta))}"
            )

t_total = time.time() - t_start
print("="*60)
print(f"DONE in {timedelta(seconds=int(t_total))}")
print(f"Results saved to: {csv_path}")

In [ ]:
# Cell 6: Stats summary

print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Range:          {RANGE_START:,} to {RANGE_END:,}")
print(f"Candidates:     {total:,} (hard residues only)")
print(f"Solved:         {solved_count:,}")
print(f"Unsolved:       {unsolved_count:,}")
print(f"Total steps:    {total_steps:,}")
if solved_count > 0:
    print(f"Avg steps/solve:{total_steps / solved_count:,.0f}")
print(f"Total time:     {timedelta(seconds=int(t_total))}")
print(f"Rate:           {total / t_total:.0f} candidates/sec")
print()

if unsolved_ns:
    print(f"WARNING: {len(unsolved_ns)} values unsolved (hit step cap):")
    for n_val in unsolved_ns[:20]:
        print(f"  n = {n_val:,}")
    if len(unsolved_ns) > 20:
        print(f"  ... and {len(unsolved_ns) - 20} more (see {unsolved_path})")
else:
    print("All candidates solved successfully!")

print()
print("Output files in /kaggle/working/:")
for f in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, f)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {f} ({size_mb:.2f} MB)")

## Downloading Results

Results are saved to `/kaggle/working/` and automatically persist as a **dataset artifact**  
when the notebook finishes (or when you save a version).

### Option 1: Save & Run All (recommended for long runs)
1. Click **Save Version** (top-right) > **Save & Run All (Commit)**
2. Close the tab — the notebook runs in the background for up to 12 hours
3. When done, go to the notebook page > **Output** tab > download the CSV files

### Option 2: Download from the Output tab
1. After the run completes, go to the **Output** tab on the notebook page
2. Click the download icon next to each file

### Option 3: Use the Kaggle API
```bash
kaggle kernels output <your-username>/erdos-straus-solver -p ./results/
```

### Continuing the search
To continue from where this run left off, update `RANGE_START` and `RANGE_END` in **Cell 4**  
and re-run. Merge the CSV files afterward to build the full dataset.

---
*Guinea Pig Trench LLC | [github.com/Commencethescourge/erdos-straus-solver](https://github.com/Commencethescourge/erdos-straus-solver)*